In [ ]:
from pathlib import Path
import pandas as pd


def load_split_csv(csv_path: str | Path) -> list[str]:
    csv_path = Path(csv_path)
    df = pd.read_csv(csv_path)

    if df.shape[1] != 1:
        raise ValueError(f"Expected 1-column CSV, got shape={df.shape} for {csv_path}")

    return df.iloc[:, 0].astype(str).tolist()


def csv_entry_to_spectral_tif(dataset_root: str | Path, rel_path: str) -> Path:
    """
    Convert a split CSV entry like:
      SCENE/PATCH/PATCH-DATA.npy
    into:
      <dataset_root>/patches/SCENE/PATCH/PATCH-SPECTRAL_IMAGE.TIF
    """
    dataset_root = Path(dataset_root)
    rel = Path(rel_path)

    filename = rel.name
    if not filename.endswith("-DATA.npy"):
        raise ValueError(f"Unexpected split filename format: {rel_path}")

    spectral_name = filename.replace("-DATA.npy", "-SPECTRAL_IMAGE.TIF")
    spectral_rel = rel.parent / spectral_name

    return dataset_root / "patches" / spectral_rel


def resolve_split_paths(dataset_root: str | Path, csv_path: str | Path) -> list[Path]:
    rel_paths = load_split_csv(csv_path)
    paths = [csv_entry_to_spectral_tif(dataset_root, p) for p in rel_paths]

    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} resolved spectral files do not exist. "
            f"First missing: {missing[0]}"
        )

    return paths

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset
import tifffile as tiff

INVALID_CHANNELS = [
    126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136,
    137, 138, 139, 140, 160, 161, 162, 163, 164, 165, 166
]

class HSITiffDataset(Dataset):
    def __init__(
        self,
        paths: list[str | Path],
        nodata_value: int = -32768,
        replace_nodata_with: float = 0.0,
        transform=None,
        return_mask: bool = False,
        invalid_channels: list[int] | None = None,
        drop_invalid_channels: bool = False,
    ):
        self.paths = [Path(p) for p in paths]
        self.nodata_value = nodata_value
        self.replace_nodata_with = replace_nodata_with
        self.transform = transform
        self.return_mask = return_mask
        self.invalid_channels = sorted(invalid_channels or [])
        self.drop_invalid_channels = drop_invalid_channels

        if len(self.paths) == 0:
            raise ValueError("Empty dataset: no paths provided.")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]

        x = tiff.imread(path)  # expected: (B, H, W)
        if x.ndim != 3:
            raise ValueError(f"Expected 3D tensor, got shape {x.shape} for {path}")

        valid_mask = (x != self.nodata_value)

        # Explicitly mark known invalid channels as fully invalid
        if self.invalid_channels:
            valid_mask[self.invalid_channels, :, :] = False

        x = x.astype(np.float32)
        x[~valid_mask] = self.replace_nodata_with

        if self.drop_invalid_channels and self.invalid_channels:
            keep = np.ones(x.shape[0], dtype=bool)
            keep[self.invalid_channels] = False
            x = x[keep]
            valid_mask = valid_mask[keep]

        x = torch.from_numpy(x)
        valid_mask = torch.from_numpy(valid_mask.astype(np.bool_))

        if self.transform is not None:
            x = self.transform(x, valid_mask)

        patch_id = path.stem.replace("-SPECTRAL_IMAGE", "")

        if self.return_mask:
            return {
                "x": x,
                "valid_mask": valid_mask,
                "path": str(path),
                "patch_id": patch_id,
            }

        return x

In [ ]:
import torch


class BandStandardize:
    def __init__(self, mean: torch.Tensor, std: torch.Tensor, eps: float = 1e-6):
        self.mean = mean.float()
        self.std = std.float()
        self.eps = eps

    def __call__(self, x: torch.Tensor, valid_mask: torch.Tensor | None = None) -> torch.Tensor:
        x = (x - self.mean[:, None, None]) / (self.std[:, None, None] + self.eps)

        if valid_mask is not None:
            x = x.clone()
            x[~valid_mask] = 0.0

        return x

In [ ]:
import torch
from torch.utils.data import DataLoader


@torch.no_grad()
def compute_band_stats(dataset, max_samples=None, num_workers: int = 0):
    """
    Compute per-band mean/std using only valid pixels.
    Expects dataset items to be dicts with keys:
      - "x": (B, H, W)
      - "valid_mask": (B, H, W)
    """
    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=num_workers,
    )

    sum_ = None
    sumsq_ = None
    count_ = None

    for i, batch in enumerate(loader):
        if max_samples is not None and i >= max_samples:
            break

        x = batch["x"].squeeze(0)          # (B, H, W)
        m = batch["valid_mask"].squeeze(0) # (B, H, W)

        B = x.shape[0]

        if sum_ is None:
            sum_ = torch.zeros(B, dtype=torch.float64)
            sumsq_ = torch.zeros(B, dtype=torch.float64)
            count_ = torch.zeros(B, dtype=torch.float64)

        for b in range(B):
            xb = x[b][m[b]]
            if xb.numel() == 0:
                continue

            sum_[b] += xb.double().sum()
            sumsq_[b] += (xb.double() ** 2).sum()
            count_[b] += xb.numel()

    mean = sum_ / torch.clamp(count_, min=1.0)
    var = sumsq_ / torch.clamp(count_, min=1.0) - mean ** 2
    var = torch.clamp(var, min=1e-12)
    std = torch.sqrt(var)

    return mean.float(), std.float(), count_

In [ ]:
import torch


def masked_mse(x_hat: torch.Tensor, x: torch.Tensor, mask: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    mask = mask.float()
    se = (x_hat - x) ** 2
    se = se * mask
    denom = mask.sum().clamp_min(eps)
    return se.sum() / denom


def masked_rmse(x_hat: torch.Tensor, x: torch.Tensor, mask: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return torch.sqrt(masked_mse(x_hat, x, mask, eps=eps))


def masked_psnr(
    x_hat: torch.Tensor,
    x: torch.Tensor,
    mask: torch.Tensor,
    data_range: float = 1.0,
    eps: float = 1e-12,
) -> torch.Tensor:
    mse = masked_mse(x_hat, x, mask, eps=eps)
    return 10.0 * torch.log10((data_range ** 2) / (mse + eps))


def masked_sam(x_hat: torch.Tensor, x: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    x_hat, x, mask: (N, B, H, W)
    SAM is computed only for pixels where all bands are valid.
    """
    pixel_mask = mask.all(dim=1)  # (N, H, W)

    x_hat = x_hat.permute(0, 2, 3, 1)  # (N, H, W, B)
    x = x.permute(0, 2, 3, 1)

    dot = torch.sum(x_hat * x, dim=-1)
    norm_hat = torch.norm(x_hat, dim=-1)
    norm = torch.norm(x, dim=-1)

    cos = dot / (norm_hat * norm + eps)
    cos = torch.clamp(cos, -1.0, 1.0)

    sam = torch.acos(cos)

    if pixel_mask.sum() == 0:
        return torch.tensor(float("nan"), device=x.device)

    return sam[pixel_mask].mean()


def masked_sam_deg(x_hat: torch.Tensor, x: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return masked_sam(x_hat, x, mask, eps=eps) * 180.0 / torch.pi

In [ ]:
import torch.nn as nn


class TinyHSIAutoencoder(nn.Module):
    def __init__(self, bands: int = 224, latent_channels: int = 16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(bands, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, latent_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_channels, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, bands, kernel_size=3, padding=1),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

In [ ]:
from pathlib import Path
import torch

dataset_root = Path("/home/brwsx/hsi-compression/pipelines/pull-dataset/hyspectnet-11k/hyspecnet-11k-full")
split_csv = dataset_root / "splits" / "easy" / "train.csv"

paths = resolve_split_paths(dataset_root, split_csv)

ds_raw = HSITiffDataset(
    paths,
    return_mask=True,
    transform=None,
)

mean, std, count = compute_band_stats(ds_raw)
valid_band_mask = count > 0

print("mean shape:", mean.shape)
print("std shape:", std.shape)
print("first 10 mean:", mean[:10])
print("first 10 std:", std[:10])
print("bands with zero valid count:", int((count == 0).sum().item()))

out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

out_path = out_dir / "band_stats_easy_train_full.pt"

torch.save(
    {
        "mean": mean,
        "std": std,
        "count": count,
        "valid_band_mask": valid_band_mask,``
        "split": str(split_csv),
        "normalization": "per_band_standardize_valid_only",
        "nodata_value": -32768,
    },
    out_path,
)

print(f"Saved stats to: {out_path.resolve()}")

In [ ]:
from pathlib import Path

dataset_root = Path("/home/brwsx/hsi-compression/pipelines/pull-dataset/hyspectnet-11k/hyspecnet-11k-full")
split_csv = dataset_root / "splits" / "easy" / "train.csv"

paths = resolve_split_paths(dataset_root, split_csv)
print("Resolved paths:", len(paths))
print("First path:", paths[0])

ds_raw = HSITiffDataset(paths, return_mask=True, transform=None)
sample = ds_raw[0]

x = sample["x"]
m = sample["valid_mask"]

print("x shape:", tuple(x.shape))
print("x dtype:", x.dtype)
print("valid ratio:", m.float().mean().item())
print("x min:", x.min().item())
print("x max:", x.max().item())
print("x mean:", x.mean().item())
print("x std:", x.std().item())

In [ ]:
from pathlib import Path
import torch

dataset_root = Path("/home/brwsx/hsi-compression/pipelines/pull-dataset/hyspectnet-11k/hyspecnet-11k-full")
split_csv = dataset_root / "splits" / "easy" / "train.csv"

stats = torch.load("artifacts/band_stats_easy_train_full.pt", map_location="cpu")
transform = BandStandardize(stats["mean"], stats["std"])

paths = resolve_split_paths(dataset_root, split_csv)
ds_norm = HSITiffDataset(paths, transform=transform, return_mask=True)

sample = ds_norm[0]
x = sample["x"]
m = sample["valid_mask"]

vals = x[m]

print("normalized x shape:", tuple(x.shape))
print("valid ratio:", m.float().mean().item())
print("valid mean:", vals.mean().item())
print("valid std:", vals.std().item())
print("all min:", x.min().item())
print("all max:", x.max().item())

In [ ]:
from pathlib import Path
import random
import torch
from torch.utils.data import DataLoader, Subset

dataset_root = Path("/home/brwsx/hsi-compression/pipelines/pull-dataset/hyspectnet-11k/hyspecnet-11k-full")
split_csv = dataset_root / "splits" / "easy" / "train.csv"

stats = torch.load("artifacts/band_stats_easy_train_full.pt", map_location="cpu")
transform = BandStandardize(stats["mean"], stats["std"])

paths = resolve_split_paths(dataset_root, split_csv)
ds_norm = HSITiffDataset(paths, transform=transform, return_mask=True)

rng = random.Random(42)
indices = rng.sample(range(len(ds_norm)), 500)

subset = Subset(ds_norm, indices)
loader = DataLoader(subset, batch_size=4, shuffle=False, num_workers=0)

all_sum = 0.0
all_sumsq = 0.0
all_count = 0

for batch in loader:
    x = batch["x"]
    m = batch["valid_mask"]

    vals = x[m]
    all_sum += vals.sum().item()
    all_sumsq += (vals ** 2).sum().item()
    all_count += vals.numel()

mean = all_sum / all_count
var = all_sumsq / all_count - mean ** 2
std = var ** 0.5

print("normalized valid mean:", mean)
print("normalized valid std:", std)


In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Subset

device = torch.device("cpu")

dataset_root = Path("/home/brwsx/hsi-compression/pipelines/pull-dataset/hyspectnet-11k/hyspecnet-11k-full")
split_csv = dataset_root / "splits" / "easy" / "train.csv"

stats = torch.load("artifacts/band_stats_easy_train_full.pt", map_location="cpu")
transform = BandStandardize(stats["mean"], stats["std"])

paths = resolve_split_paths(dataset_root, split_csv)
ds_norm = HSITiffDataset(paths, transform=transform, return_mask=True)

small_ds = Subset(ds_norm, list(range(8)))
loader = DataLoader(small_ds, batch_size=2, shuffle=True, num_workers=0)

model = TinyHSIAutoencoder(bands=224, latent_channels=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):
    model.train()
    total_loss = 0.0

    for batch in loader:
        x = batch["x"].to(device)
        mask = batch["valid_mask"].to(device)

        optimizer.zero_grad()
        x_hat = model(x)
        loss = masked_mse(x_hat, x, mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch + 1:03d} | masked_loss={avg_loss:.6f}")